# FDGAN: sample from a trained checkpoint

This notebook samples Traffic Signs images from a trained FDGAN checkpoint with the settings of the paper
(mask channel on, the perturbed-input prediction inside the band, mask guidance 2.0, 500 ancestral steps).
It needs a GPU runtime (Runtime > Change runtime type > T4 or better); 16 images take a few minutes on a T4.

What it does:
1. installs the repository and its two extra packages;
2. fetches the Traffic Signs crops of our dataset release and writes one box file per crop;
3. loads a checkpoint that you point it to;
4. samples 16 images, one per box, and shows them with the boxes drawn.

Training a model from scratch takes about a day on an A100 and is not something to do in Colab; the job files in
`jobs/` show every flag of the paper's runs, and the last cell runs a short smoke training on the same data if you
want to see the loop go.

In [ ]:
# 1. install
!apt-get -qq install -y libopenmpi-dev openmpi-bin > /dev/null
!pip -q install blobfile mpi4py
!git clone -q https://github.com/Latzi/focused-diffusion-gan.git /content/focused-diffusion-gan
%cd /content/focused-diffusion-gan
!pip -q install -e . > /dev/null
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 2. Traffic Signs crops and boxes (about 40 MB)
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Latzi/object-centric-low-data-datasets.git /content/ocld
!cd /content/ocld && git sparse-checkout set traffic_signs/Train_Data > /dev/null
!python tools/make_box_json_from_release.py --traffic_signs /content/ocld/traffic_signs \
    --out_boxes /content/data/traffic_bb --out_images /content/data/traffic_img

## 3. The checkpoint

Set `CHECKPOINT` to an EMA checkpoint of a Traffic Signs model (`ema_0.9999_200000.pt`, about 460 MB). Either a
direct download link, or a path on your Google Drive (the cell mounts the drive when the path starts with
`/content/drive`).

In [ ]:
CHECKPOINT = ""   # e.g. "https://.../ema_0.9999_200000.pt" or "/content/drive/MyDrive/FDGAN/ema_0.9999_200000.pt"

import os, urllib.request
assert CHECKPOINT, "set CHECKPOINT first"
if CHECKPOINT.startswith("http"):
    os.makedirs("/content/ckpt", exist_ok=True)
    CKPT = "/content/ckpt/" + os.path.basename(CHECKPOINT.split("?")[0])
    if not os.path.exists(CKPT):
        urllib.request.urlretrieve(CHECKPOINT, CKPT)
else:
    if CHECKPOINT.startswith("/content/drive") and not os.path.ismount("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
    CKPT = CHECKPOINT
assert os.path.exists(CKPT), f"not found: {CKPT}"
print("checkpoint:", CKPT, f"{os.path.getsize(CKPT)/1e6:.0f} MB")

In [ ]:
# 4. sample 16 images, one per box (boxes in sorted file order, seed 17, as in the paper's sets)
!python scripts/image_sample_fdgan_4ch.py --model_path "$CKPT" --bb_dir /content/data/traffic_bb --out_dir /content/samples \
    --num_samples 16 --batch_size 8 --seed 17 --timestep_respacing 500 --mask_cfg_scale 2.0 \
    --image_size 256 --num_channels 128 --num_res_blocks 2 --num_heads 1 --attention_resolutions 16 --use_scale_shift_norm False --resblock_updown False --learn_sigma True --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False \
    --use_bb_mask_cond True --mask_gain 50.0 --mask_mode always --roi_bump_mode predict --k_focus 1.75 --bump_tmin_rev 700 --bump_tmax_rev 820 --bb_is_xywh True --bb_xyxy_end_inclusive False

In [ ]:
# 5. look at them
import json, matplotlib.pyplot as plt
from PIL import Image, ImageDraw
meta = json.load(open("/content/samples/meta.json"))
fig, axes = plt.subplots(2, 8, figsize=(20, 5.4))
for ax, rec in zip(axes.flat, meta["samples"]):
    im = Image.open(f"/content/samples/{rec['file']}").convert("RGB")
    x0, y0, x1, y1 = rec["bb_xyxy"]
    ImageDraw.Draw(im).rectangle([x0, y0, x1 - 1, y1 - 1], outline=(255, 40, 40), width=2)
    ax.imshow(im); ax.set_title(rec["file"][-10:-4], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
print("cost:", meta["cost"])

## Optional: a short training run on the same data

200 iterations with batch 4, just to see the loop, the logs and the checkpoint files appear. The paper's runs use the
flags in `jobs/train_traffic2_fdgan_mask.sh` (batch 16, 200,000 iterations, adversarial stage from iteration 40,000).

In [ ]:
!python scripts/image_train_Run2_3_10_FDGAN_COCO_LateBand.py --data_dir /content/data/traffic_img --bb_dir /content/data/traffic_bb \
    --log_dir /content/smoke/logs --checkpoint_dir /content/smoke/ckpt --image_save_dir /content/smoke/img \
    --image_size 256 --num_channels 128 --num_res_blocks 2 --num_heads 1 --attention_resolutions 16 --use_scale_shift_norm False --resblock_updown False --learn_sigma True --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --class_cond False --dropout 0.0 \
    --lr 1e-4 --batch_size 4 --microbatch 4 --use_fp16 True --ema_rate 0.9999 --log_interval 10 --save_interval 200 --lr_anneal_steps 200 \
    --use_bb_mask_cond True --bb_is_xywh True --bb_xyxy_end_inclusive False --gan_warmup_steps 100 \
    --tmin_start 820 --tmin_end 700 --band_width 120 --tmax_cap 950 --band_ramp_steps 50000 --diffusion_in_band_weight 0.7 \
    --gan_initial_scale 1e-4 --gan_final_scale 1e-2 --l1_initial_scale 5e-2 --l1_final_scale 5e-2 --lambda_enhanced 5e-2 \
    --color_stats_scale 0.05 --sat_penalty_scale 0.01 --sat_threshold 0.95 --noise_option extra_gan_only --k_focus 1.75 --max_gan_per_microbatch 4 \
    --use_mask_adapter True --ndf 128 --n_layers_D 3 --lr_D 1e-5 --beta1_D 0.5 --beta2_D 0.999 --mask_gain 50.0 --mask_init_mode copy_mean --mask_init_scale 0.10
!ls -la /content/smoke/ckpt